In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
out_dir = "../data/processed/Restaurant Artifacts"
items_path = f"{out_dir}/restaurant_items.parquet"
reviews_path = "../data/processed/User Artifacts/filtered_reviews.csv"

# Load Items
items_tbl = pq.read_table(items_path)
items_df = items_tbl.to_pandas()

# Load Reviews
reviews = pd.read_csv(reviews_path)
print(items_df.shape, reviews.shape)

In [ ]:
# Convert strings to Datetime
if "checkin_date" in reviews.columns and "checkin_hour" not in reviews.columns:
    reviews["checkin_date"] = pd.to_datetime(reviews["checkin_date"], errors="coerce")
    reviews["checkin_hour"] = reviews["checkin_date"].dt.hour
    reviews["checkin_dow"] = reviews["checkin_date"].dt.weekday

# basic context features
reviews["weekend"] = (reviews["checkin_dow"] >= 5).astype(np.int64)

In [ ]:
# Only use businesses that exist in items_df
biz_set = set(items_df["business_id"].astype(str))
reviews["business_id"] = reviews["business_id"].astype(str)
reviews["user_id"] = reviews["user_id"].astype(str)

reviews = reviews[reviews["business_id"].isin(biz_set)].copy()

# Positives Baseline
reviews["label"] = (reviews["stars"] >= 3.5).astype(np.int64)
reviews_pos = reviews[reviews["label"] == 1].copy()

# Split by User
users = np.array(reviews_pos["user_id"].unique(), copy=True)
rng = np.random.default_rng(42)
rng.shuffle(users)

# Train, Val, Test Split
n = len(users)
train_users = set(users[: int(0.8*n)])
val_users   = set(users[int(0.8*n): int(0.9*n)])
test_users  = set(users[int(0.9*n):])

train_df = reviews_pos[reviews_pos["user_id"].isin(train_users)].copy()
val_df   = reviews_pos[reviews_pos["user_id"].isin(val_users)].copy()
test_df  = reviews_pos[reviews_pos["user_id"].isin(test_users)].copy()

print(train_df.shape, val_df.shape, test_df.shape)

In [ ]:
# Index items by business_id
items_df["business_id"] = items_df["business_id"].astype(str)
items_df = items_df.drop_duplicates("business_id").set_index("business_id")

# Vocab sizes
# Category_id = around 500
cat_vocab_size = int(max([max(x) for x in items_df["category_ids"]]) + 1)
region_vocab_size = int(items_df["region_id"].max() + 1)
price_vocab_size = 5
attr_dim = len(items_df.iloc[0]["attr_vec"])

print("cat_vocab_size:", cat_vocab_size)
print("region_vocab_size:", region_vocab_size)
print("attr_dim:", attr_dim)

In [ ]:
def hour_to_bucket(h):
    # 6 buckets: late-night, breakfast, lunch, afternoon, dinner, night
    if h < 5:   return 0
    if h < 11:  return 1
    if h < 14:  return 2
    if h < 17:  return 3
    if h < 21:  return 4
    return 5

for df_ in [train_df, val_df, test_df]:
    df_["time_bucket"] = df_["checkin_hour"].apply(hour_to_bucket).astype(np.int64)
    df_["dow"] = df_["checkin_dow"].astype(np.int64)
    df_["weekend"] = df_["weekend"].astype(np.int64)

In [ ]:
# User ID Mapping
"""all_users = pd.Index(train_df["user_id"].unique())
user2idx = {u:i for i,u in enumerate(all_users)}"""
all_users = pd.Index(pd.concat([train_df["user_id"], val_df["user_id"], test_df["user_id"]]).unique())
user2idx = {u:i for i,u in enumerate(all_users)}
num_users = len(user2idx)
print("num_users:", num_users)

def map_user(df_):
    df_ = df_.copy()
    df_["user_idx"] = df_["user_id"].map(user2idx)
    return df_.dropna(subset=["user_idx"]).astype({"user_idx": "int64"})

train_df = map_user(train_df)
val_df   = map_user(val_df)
test_df  = map_user(test_df)

print(train_df.shape, val_df.shape, test_df.shape)

In [ ]:
class PairDataset(Dataset):
    def __init__(self, interactions_df, items_df_indexed):
        self.df = interactions_df.reset_index(drop=True)
        # Business_id indexed
        self.items = items_df_indexed

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        bid = r["business_id"]
        item = self.items.loc[bid]
        return {
            "user_idx": int(r["user_idx"]),
            "time_bucket": int(r["time_bucket"]),
            "dow": int(r["dow"]),
            "weekend": int(r["weekend"]),
            "cat_ids": item["category_ids"],
            "price": int(item["price_tier"]),
            "region": int(item["region_id"]),
            "attr": item["attr_vec"]}

def collate_pairs(batch):
    B = len(batch)

    # user/context
    user_idx = torch.tensor([b["user_idx"] for b in batch], dtype=torch.long)
    time_bucket = torch.tensor([b["time_bucket"] for b in batch], dtype=torch.long)
    dow = torch.tensor([b["dow"] for b in batch], dtype=torch.long)
    weekend = torch.tensor([b["weekend"] for b in batch], dtype=torch.float32).unsqueeze(-1)

    # Pad category ids
    cat_lists = [b["cat_ids"] for b in batch]
    L = max(len(x) for x in cat_lists)
    cat_ids = torch.zeros((B, L), dtype=torch.long)
    cat_mask = torch.zeros((B, L), dtype=torch.float32)
    for i, ids in enumerate(cat_lists):
        ids = [int(v) for v in ids]
        cat_ids[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        cat_mask[i, :len(ids)] = 1.0

    price = torch.tensor([b["price"] for b in batch], dtype=torch.long)
    region = torch.tensor([b["region"] for b in batch], dtype=torch.long)
    attr = torch.from_numpy(np.stack([b["attr"] for b in batch],
                                     axis=0)).float()

    return {"user_idx": user_idx,
        "time_bucket": time_bucket,
        "dow": dow,
        "weekend": weekend,
        "cat_ids": cat_ids,
        "cat_mask": cat_mask,
        "price": price,
        "region": region,
        "attr": attr}

train_loader = DataLoader(PairDataset(train_df, items_df), batch_size=512, shuffle=True, collate_fn=collate_pairs)
val_loader   = DataLoader(PairDataset(val_df, items_df), batch_size=512, shuffle=False, collate_fn=collate_pairs)

In [ ]:
print("train_df:", len(train_df), "val_df:", len(val_df), "test_df:", len(test_df))
print("len(train_loader):", len(train_loader), "len(val_loader):", len(val_loader))

In [ ]:
# Item Tower
class ItemTower(nn.Module):
    def __init__(self, cat_vocab_size, region_vocab_size, price_vocab_size, attr_dim, d=128):
        super().__init__()
        self.d = d
        self.cat_emb = nn.Embedding(cat_vocab_size, d)
        # Price
        self.price_emb = nn.Embedding(price_vocab_size, 16)
        # Region
        self.region_emb = nn.Embedding(region_vocab_size, 16)
        # Attribute
        self.attr_proj = nn.Sequential(nn.Linear(attr_dim, 64), nn.ReLU())

        self.mlp = nn.Sequential(
            nn.Linear(d + 16 + 16 + 64, 256),
            nn.ReLU(),
            nn.Linear(256, d))

    def forward(self, cat_ids, cat_mask, price, region, attr):
        # Cat pooling, Shape: (B, L, d)
        ce = self.cat_emb(cat_ids)
        # Shape (B, L, 1)
        m = cat_mask.unsqueeze(-1)
        # (B, d)
        pooled = (ce * m).sum(1) / m.sum(1).clamp_min(1.0)

        x = torch.cat([pooled, self.price_emb(price), self.region_emb(region), self.attr_proj(attr)], dim=-1)
        v = self.mlp(x)
        return F.normalize(v, p=2, dim=-1)

In [ ]:
# User/Context Tower (w/ Contextual Features)
class UserTower(nn.Module):
    def __init__(self, num_users, region_vocab_size=None, d=128):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, 64)
        self.time_emb = nn.Embedding(6, 16)
        self.dow_emb  = nn.Embedding(7, 8)

        self.mlp = nn.Sequential(
            nn.Linear(64 + 16 + 8 + 1, 256),
            nn.ReLU(),
            nn.Linear(256, d),
        )

    def forward(self, user_idx, time_bucket, dow, weekend):
        x = torch.cat([
            self.user_emb(user_idx),
            self.time_emb(time_bucket),
            self.dow_emb(dow),
            weekend,  # float (B,1)
        ], dim=-1)
        u = self.mlp(x)
        return F.normalize(u, p=2, dim=-1)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

item_tower = ItemTower(cat_vocab_size, region_vocab_size, price_vocab_size, attr_dim, d=128).to(device)
user_tower = UserTower(num_users, d=128).to(device)

opt = torch.optim.Adam(list(item_tower.parameters()) + list(user_tower.parameters()), lr=1e-3)

def train_one_epoch(loader, tau=0.1):
    item_tower.train(); user_tower.train()
    total = 0.0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        u = user_tower(batch["user_idx"], batch["time_bucket"], batch["dow"], batch["weekend"])
        v = item_tower(batch["cat_ids"], batch["cat_mask"], batch["price"], batch["region"], batch["attr"])

        logits = (u @ v.T) / tau  # (B,B)
        labels = torch.arange(logits.size(0), device=device)
        loss = F.cross_entropy(logits, labels)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total += loss.item()
    return total / max(1, len(loader))

@torch.no_grad()
def eval_loss(loader, tau=0.1):
    item_tower.eval(); user_tower.eval()
    total = 0.0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        u = user_tower(batch["user_idx"], batch["time_bucket"], batch["dow"], batch["weekend"])
        v = item_tower(batch["cat_ids"], batch["cat_mask"], batch["price"], batch["region"], batch["attr"])
        logits = (u @ v.T) / tau
        labels = torch.arange(logits.size(0), device=device)
        loss = F.cross_entropy(logits, labels)
        total += loss.item()
    return total / max(1, len(loader))

for epoch in range(5):
    tr = train_one_epoch(train_loader)
    va = eval_loss(val_loader)
    print(f"epoch {epoch+1} train_loss={tr:.4f} val_loss={va:.4f}")

Parameters to Adjust
1. Main embedding dimension, d = 128
2. Vocab Sizes: cat_vocab_size, region_vocab_size, price_vocab_size
3. Embeding Sizes
4. MLP
5. Pooling Strategy (Mean atm )
6. Final Normalization (F. normalize)